# Nichols Chart

The Nichols chart represents the open-loop frequency response $G(j\omega)$ in a \
**phase–magnitude plane**. As $\omega$ sweeps from $0$ to $\infty$, the curve traces a path where:

- **x-axis:** $\angle G(j\omega)$ — phase in degrees
- **y-axis:** $|G(j\omega)|_{\mathrm{dB}} = 20\log_{10}|G(j\omega)|$

For the second-order plant with one finite zero studied here:

$$G(s) = K\,\frac{s - z}{(s - p_1)(s - p_2)}$$

The **M-contours** (gray curves) are iso-curves of constant closed-loop magnitude \
$\left|\tfrac{G}{1+G}\right|_{\!\mathrm{dB}} = M$, making bandwidth and resonance peaks \
directly readable from the open-loop locus — without computing the closed-loop transfer function. \
Frequency is an implicit parameter; triangle and square markers indicate low- and high-$\omega$ endpoints.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

%matplotlib inline


def _eval_G(K, z, p1, p2, omega):
    s = 1j * omega
    H = K * (s - z) / ((s - p1) * (s - p2))
    return 20 * np.log10(np.abs(H) + 1e-15), np.degrees(np.unwrap(np.angle(H)))


def _m_contours(ax):
    ph = np.linspace(-360, 0, 700)
    mg = np.linspace(-50, 50, 700)
    PH, MG = np.meshgrid(ph, mg)
    G = 10 ** (MG / 20) * np.exp(1j * np.radians(PH))
    Tdb = 20 * np.log10(np.abs(G / (1 + G)) + 1e-15)
    cs = ax.contour(PH, MG, Tdb,
                    levels=[-20, -12, -6, -3, -1, 0, 1, 3, 6, 12],
                    colors='gray', linewidths=0.7, alpha=0.55)
    ax.clabel(cs, fmt='%d dB', fontsize=7)


def _margins(mag, phi, omega):
    GM = PM = wpc = wgc = None
    p180 = phi + 180
    sc = np.where(np.diff(np.sign(p180)))[0]
    if sc.size:
        i = sc[0]
        t   = -p180[i] / (p180[i + 1] - p180[i] + 1e-15)
        wpc = omega[i] + t * (omega[i + 1] - omega[i])
        GM  = -(mag[i]  + t * (mag[i + 1]  - mag[i]))
    gsc = np.where(np.diff(np.sign(mag)))[0]
    if gsc.size:
        i = gsc[0]
        t   = -mag[i] / (mag[i + 1] - mag[i] + 1e-15)
        wgc = omega[i] + t * (omega[i + 1] - omega[i])
        PM  = phi[i]  + t * (phi[i + 1]  - phi[i]) + 180
    return GM, PM, wpc, wgc


def plot_nichols(K=1.0, zero=1.0, pole1=-1.0, pole2=-2.0):
    omega = np.logspace(-2, 2, 3000)
    mag, phi = _eval_G(K, zero, pole1, pole2, omega)
    GM, PM, *_ = _margins(mag, phi, omega)

    fig, ax = plt.subplots(figsize=(8, 6))
    _m_contours(ax)
    ax.plot(phi, mag, 'b', lw=2)
    ax.plot(phi[0],  mag[0],  'g^', ms=9, label='low \u03c9')
    ax.plot(phi[-1], mag[-1], 'rs', ms=9, label='high \u03c9')
    ax.plot(-180, 0, 'k+', ms=14, mew=2.5, label='critical point')

    ax.set_xlim(-360, 0)
    ax.set_ylim(-50, 50)
    ax.axvline(-180, color='k', ls='--', lw=0.8, alpha=0.5)
    ax.axhline(0,    color='k', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlabel('Phase  (deg)')
    ax.set_ylabel('Magnitude  (dB)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    gm_s = f'{GM:.1f} dB' if GM is not None else '\u221e'
    pm_s = f'{PM:.1f}\u00b0' if PM is not None else '\u221e'
    stable = (GM is None or GM > 0) and (PM is None or PM > 0)
    ax.set_title(
        f'Nichols Chart   GM = {gm_s}   PM = {pm_s}'
        f'   [{"Stable" if stable else "Unstable"}]',
        color='green' if stable else 'red', fontsize=12)
    plt.tight_layout()
    plt.show()


interact(plot_nichols,
         K    =FloatSlider(min=0.1, max=10, step=0.1, value=1.0,  description='K'),
         zero =FloatSlider(min=-5,  max=5,  step=0.1, value=1.0,  description='zero'),
         pole1=FloatSlider(min=-5,  max=0,  step=0.1, value=-1.0, description='pole\u2081'),
         pole2=FloatSlider(min=-5,  max=0,  step=0.1, value=-2.0, description='pole\u2082'))

interactive(children=(FloatSlider(value=1.0, description='K', max=10.0, min=0.1), FloatSlider(value=1.0, descr…

<function __main__.plot_nichols(K=1.0, zero=1.0, pole1=-1.0, pole2=-2.0)>

## Stability Margins

**Gain Margin (GM)** — measured at the **phase crossover frequency** $\omega_{pc}$, \
where $\angle G = -180°$:

$$GM = -\,|G(j\omega_{pc})|_{\mathrm{dB}}$$

A positive GM means the gain can increase by that many dB before the closed-loop system becomes \
unstable. On the Nichols chart, GM is the **vertical distance** from the curve to the 0 dB axis \
at $\phi = -180°$ (red arrow below).

**Phase Margin (PM)** — measured at the **gain crossover frequency** $\omega_{gc}$, \
where $|G| = 0\,\mathrm{dB}$:

$$PM = \angle G(j\omega_{gc}) + 180°$$

PM is the **horizontal distance** from the curve to the $-180°$ axis at 0 dB (green arrow). \
For minimum-phase systems, closed-loop stability requires $GM > 0$ and $PM > 0$ — equivalently, \
the critical point $(-180°,\,0\,\mathrm{dB})$ must lie below-left of the open-loop curve. \
The Bode diagrams confirm both margins in the frequency domain.

In [ ]:
import matplotlib.gridspec as gridspec


def plot_margins(K=1.0, zero=1.0, pole1=-1.0, pole2=-2.0):
    omega = np.logspace(-2, 2, 3000)
    mag, phi = _eval_G(K, zero, pole1, pole2, omega)
    GM, PM, wpc, wgc = _margins(mag, phi, omega)

    fig = plt.figure(figsize=(13, 6))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
    axm = fig.add_subplot(gs[0, 0])
    axp = fig.add_subplot(gs[1, 0], sharex=axm)
    axn = fig.add_subplot(gs[:, 1])

    # Bode – magnitude
    axm.semilogx(omega, mag, 'b', lw=2)
    axm.axhline(0, color='k', ls='--', lw=0.8)
    axm.set_ylabel('Mag (dB)')
    axm.set_title('Bode Diagram')
    axm.grid(True, which='both', alpha=0.3)
    if wgc is not None:
        axm.axvline(wgc, color='g', ls=':', lw=1.2)
        axm.plot(wgc, 0, 'go', ms=7)
    if wpc is not None and GM is not None:
        idx = np.argmin(np.abs(omega - wpc))
        axm.axvline(wpc, color='r', ls=':', lw=1.2)
        axm.annotate('', xy=(wpc, 0), xytext=(wpc, mag[idx]),
                     arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
        axm.text(wpc * 1.15, mag[idx] / 2, f'GM={GM:.1f} dB', color='r', fontsize=8)

    # Bode – phase
    axp.semilogx(omega, phi, 'b', lw=2)
    axp.axhline(-180, color='k', ls='--', lw=0.8)
    axp.set_ylabel('Phase (deg)')
    axp.set_xlabel('Frequency (rad/s)')
    axp.grid(True, which='both', alpha=0.3)
    if wpc is not None:
        axp.axvline(wpc, color='r', ls=':', lw=1.2)
        axp.plot(wpc, -180, 'ro', ms=7)
    if wgc is not None and PM is not None:
        idx = np.argmin(np.abs(omega - wgc))
        axp.axvline(wgc, color='g', ls=':', lw=1.2)
        axp.annotate('', xy=(wgc, -180), xytext=(wgc, phi[idx]),
                     arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
        axp.text(wgc * 1.15, (phi[idx] - 180) / 2 - 180,
                 f'PM={PM:.1f}\u00b0', color='g', fontsize=8)

    # Nichols chart
    _m_contours(axn)
    axn.plot(phi, mag, 'b', lw=2)
    axn.plot(phi[0],  mag[0],  'g^', ms=8)
    axn.plot(phi[-1], mag[-1], 'rs', ms=8)
    axn.plot(-180, 0, 'k+', ms=14, mew=2.5)
    if wpc is not None and GM is not None:
        idx = np.argmin(np.abs(omega - wpc))
        axn.annotate('', xy=(-180, 0), xytext=(-180, mag[idx]),
                     arrowprops=dict(arrowstyle='<->', color='red', lw=1.8))
        axn.text(-175, mag[idx] / 2, 'GM', color='r', fontsize=9, fontweight='bold')
    if wgc is not None and PM is not None:
        idx = np.argmin(np.abs(omega - wgc))
        axn.annotate('', xy=(-180, 0), xytext=(phi[idx], 0),
                     arrowprops=dict(arrowstyle='<->', color='green', lw=1.8))
        axn.text((phi[idx] - 180) / 2 - 180, 2.5, 'PM',
                 color='g', fontsize=9, fontweight='bold')
    axn.set_xlim(-360, 0)
    axn.set_ylim(-50, 50)
    axn.axvline(-180, color='k', ls='--', lw=0.8, alpha=0.5)
    axn.axhline(0,    color='k', ls='--', lw=0.8, alpha=0.5)
    axn.set_xlabel('Phase (deg)')
    axn.set_ylabel('Magnitude (dB)')
    axn.set_title('Nichols Chart')
    axn.grid(True, alpha=0.3)

    gm_s = f'{GM:.1f} dB' if GM is not None else '\u221e'
    pm_s = f'{PM:.1f}\u00b0' if PM is not None else '\u221e'
    stable = (GM is None or GM > 0) and (PM is None or PM > 0)
    fig.suptitle(
        f'GM = {gm_s}   |   PM = {pm_s}   |   {"Stable" if stable else "Unstable"}',
        color='green' if stable else 'red', fontsize=13, fontweight='bold')
    plt.show()


interact(plot_margins,
         K    =FloatSlider(min=0.1, max=10, step=0.1, value=1.0,  description='K'),
         zero =FloatSlider(min=-5,  max=5,  step=0.1, value=1.0,  description='zero'),
         pole1=FloatSlider(min=-5,  max=0,  step=0.1, value=-1.0, description='pole\u2081'),
         pole2=FloatSlider(min=-5,  max=0,  step=0.1, value=-2.0, description='pole\u2082'))

interactive(children=(FloatSlider(value=1.0, description='K', max=10.0, min=0.1), FloatSlider(value=1.0, descr…

<function __main__.plot_margins(K=1.0, zero=1.0, pole1=-1.0, pole2=-2.0)>